In [ ]:
# ========== 1. IMPORTS ========== #
import os
import glob
import time
import random
import shutil
import tempfile
from collections import defaultdict
from types import SimpleNamespace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm_lib
import seaborn as sns
import tensorflow as tf

from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.layers import (
    Input, Dense, GlobalAveragePooling2D, Dropout, BatchNormalization,
    Conv2D, Reshape, Multiply, Add, Activation, Layer, Permute,
    LayerNormalization, DepthwiseConv2D
)
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger
from tensorflow.keras.regularizers import l2

from sklearn.utils import class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix,
    top_k_accuracy_score, roc_curve, auc,
    cohen_kappa_score, matthews_corrcoef,
    balanced_accuracy_score
)
from sklearn.preprocessing import label_binarize
from sklearn.manifold import TSNE

# ========== 2. GPU CONFIG ========== #
print("TensorFlow version:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print("GPUs available:", len(gpus))
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU memory growth enabled.")
    except RuntimeError as e:
        print(e)

# ========== 3. MIXED PRECISION ========== #
tf.keras.mixed_precision.set_global_policy("mixed_float16")
print("Compute dtype :", tf.keras.mixed_precision.global_policy().compute_dtype)
print("Variable dtype:", tf.keras.mixed_precision.global_policy().variable_dtype)

In [ ]:
# ========== 4. EXPERIMENT CONFIGURATION ========== #

# --- Paths ---
DATA_DIR        = "/kaggle/input/datasets/giaphuc/dataset-garlic-2106/dataset_final_2006"
BASE_RESULT_DIR = "/kaggle/working/report_ConvNeXtBase_EEMA_FocalLoss_MultiRun"
os.makedirs(BASE_RESULT_DIR, exist_ok=True)

# --- Model ---
INPUT_SHAPE = (384, 384, 3)   # ConvNeXt-Base optimal resolution
BATCH_SIZE  = 32              # ConvNeXt-Base is larger than EfficientNetB4
EPOCHS      = 30

# --- Focal Loss hyperparams ---
FOCAL_GAMMA = 2.0
FOCAL_ALPHA = None   # Will be set to class-balanced weights automatically

# --- Multi-run settings ---
RANDOM_SEEDS = [42, 123, 456]

# --- Performance knobs ---
AUTOTUNE = tf.data.AUTOTUNE
tf.config.optimizer.set_jit(True)   # XLA JIT

# --- Result storage ---
all_runs_results = []

print(f"Dataset    : {DATA_DIR}")
print(f"Output dir : {BASE_RESULT_DIR}")
print(f"Input shape: {INPUT_SHAPE}")
print(f"Batch size : {BATCH_SIZE}")
print(f"Seeds      : {RANDOM_SEEDS}")
print(f"Focal Loss : gamma={FOCAL_GAMMA}")
print(f"XLA JIT    : ON")

In [ ]:
# ========== 5. E-EMA MODULE (Enhanced Efficient Multi-Scale Attention) ========== #

class EEMA(Layer):
    """Enhanced Efficient Multi-Scale Attention (E-EMA).
    
    Builds upon EMA with three enhancements:
    1. Learnable Temperature Scaling - calibrates attention sharpness
    2. Channel-Aware Gating - semantic selectivity across feature channels  
    3. Strip Convolution - preserves directional spatial morphology
    
    Args:
        channels: Number of input channels
        groups: Number of channel groups for multi-scale processing
        strip_kernel: Kernel size for strip (directional) convolutions
    """
    
    def __init__(self, channels, groups=8, strip_kernel=7, **kwargs):
        super(EEMA, self).__init__(**kwargs)
        self.channels = channels
        self.groups = groups
        self.strip_kernel = strip_kernel
        self.group_channels = channels // groups
        
    def build(self, input_shape):
        C = self.channels
        G = self.groups
        gc = self.group_channels
        sk = self.strip_kernel
        
        # --- EMA base: multi-scale depthwise convolutions per group ---
        # 1x1 channel mixing within groups
        self.group_conv = Conv2D(C, 1, padding='same', use_bias=False,
                                 groups=G, name='group_conv')
        self.group_norm = LayerNormalization(epsilon=1e-6, name='group_norm')
        
        # Multi-scale spatial: 3x3, 5x5 depthwise
        self.dw_conv3 = DepthwiseConv2D(3, padding='same', use_bias=False,
                                         name='dw_conv3x3')
        self.dw_conv5 = DepthwiseConv2D(5, padding='same', use_bias=False,
                                         name='dw_conv5x5')
        
        # --- Enhancement 1: Learnable Temperature Scaling ---
        # temperature > 0, initialized to 1.0 (neutral)
        self.temperature = self.add_weight(
            name='temperature',
            shape=(1, 1, 1, C),
            initializer=tf.keras.initializers.Ones(),
            trainable=True,
            constraint=tf.keras.constraints.NonNeg()
        )
        
        # --- Enhancement 2: Channel-Aware Gating ---
        # Squeeze-and-excitation style gating
        reduction = max(C // 16, 4)
        self.gate_squeeze = Dense(reduction, activation='relu', name='gate_squeeze')
        self.gate_excite = Dense(C, activation='sigmoid', name='gate_excite')
        
        # --- Enhancement 3: Strip Convolutions (horizontal + vertical) ---
        # Captures elongated disease lesion patterns
        self.strip_h = Conv2D(C, (1, sk), padding='same', use_bias=False,
                              groups=G, name='strip_horizontal')
        self.strip_v = Conv2D(C, (sk, 1), padding='same', use_bias=False,
                              groups=G, name='strip_vertical')
        
        # Final projection
        self.proj = Conv2D(C, 1, padding='same', use_bias=False, name='proj')
        self.proj_norm = LayerNormalization(epsilon=1e-6, name='proj_norm')
        
        super(EEMA, self).build(input_shape)
    
    def call(self, x, training=None):
        residual = x  # (B, H, W, C)
        
        # --- Multi-scale spatial attention (EMA base) ---
        # Group convolution for channel interaction
        out = self.group_conv(x)
        out = self.group_norm(out)
        
        # Multi-scale depthwise: capture patterns at 3x3 and 5x5
        ms3 = self.dw_conv3(out)
        ms5 = self.dw_conv5(out)
        multi_scale = ms3 + ms5  # fuse multi-scale features
        
        # --- Enhancement 3: Strip convolutions ---
        strip_feat = self.strip_h(out) + self.strip_v(out)
        multi_scale = multi_scale + strip_feat
        
        # --- Enhancement 1: Learnable Temperature Scaling ---
        # Sharpen or soften attention based on learned temperature
        temperature = tf.nn.softplus(self.temperature) + 1e-6  # ensure > 0
        attn_map = multi_scale / temperature
        attn_map = tf.nn.sigmoid(attn_map)  # spatial attention weights
        
        # Apply spatial attention
        out = x * attn_map
        
        # --- Enhancement 2: Channel-Aware Gating ---
        # Global context for channel gating
        gap = tf.reduce_mean(out, axis=[1, 2])  # (B, C)
        gate = self.gate_squeeze(gap)
        gate = self.gate_excite(gate)  # (B, C)
        gate = tf.reshape(gate, [-1, 1, 1, self.channels])  # (B, 1, 1, C)
        out = out * gate
        
        # Final projection + residual
        out = self.proj(out)
        out = self.proj_norm(out)
        out = out + residual
        
        return out
    
    def get_config(self):
        config = super(EEMA, self).get_config()
        config.update({
            'channels': self.channels,
            'groups': self.groups,
            'strip_kernel': self.strip_kernel,
        })
        return config


print("E-EMA module defined.")
print("  Enhancements:")
print("    1. Learnable Temperature Scaling")
print("    2. Channel-Aware Gating")
print("    3. Strip Convolution (horizontal + vertical)")

In [ ]:
# ========== 6. FOCAL LOSS ========== #

class FocalLoss(tf.keras.losses.Loss):
    """Focal Loss for multi-class classification.
    
    FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)
    
    Focuses learning on hard, misclassified samples and down-weights
    easy, well-classified examples. Particularly effective for
    class-imbalanced datasets.
    
    Args:
        gamma: Focusing parameter (default=2.0). Higher gamma = more focus on hard samples.
        alpha: Per-class balancing weights. If None, uniform weights.
        label_smoothing: Optional label smoothing factor.
    """
    
    def __init__(self, gamma=2.0, alpha=None, label_smoothing=0.0,
                 name='focal_loss', **kwargs):
        super().__init__(name=name, **kwargs)
        self.gamma = gamma
        self.alpha = alpha
        self.label_smoothing = label_smoothing
    
    def call(self, y_true, y_pred):
        # y_true: one-hot (B, C), y_pred: softmax probabilities (B, C)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        
        # Label smoothing
        if self.label_smoothing > 0:
            num_classes = tf.cast(tf.shape(y_true)[-1], tf.float32)
            y_true = y_true * (1.0 - self.label_smoothing) + \
                     self.label_smoothing / num_classes
        
        # Cross-entropy component
        ce = -y_true * tf.math.log(y_pred)
        
        # Focal modulation: (1 - p_t)^gamma
        p_t = tf.reduce_sum(y_true * y_pred, axis=-1, keepdims=True)
        focal_weight = tf.pow(1.0 - p_t, self.gamma)
        
        # Apply focal weight
        focal_loss = focal_weight * ce
        
        # Apply class-balancing alpha
        if self.alpha is not None:
            alpha_tensor = tf.constant(self.alpha, dtype=tf.float32)
            alpha_tensor = tf.reshape(alpha_tensor, [1, -1])
            focal_loss = focal_loss * alpha_tensor
        
        return tf.reduce_mean(tf.reduce_sum(focal_loss, axis=-1))
    
    def get_config(self):
        config = super().get_config()
        config.update({
            'gamma': self.gamma,
            'alpha': self.alpha,
            'label_smoothing': self.label_smoothing,
        })
        return config


print("Focal Loss defined.")
print(f"  gamma={FOCAL_GAMMA} — penalizes easy samples more aggressively")
print(f"  alpha=class-balanced (computed from training distribution)")

In [ ]:
# ========== 7. HELPER FUNCTIONS ========== #

# ---------- GPU-side augmentation ----------
_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.083),        # ≈ ±30°
    tf.keras.layers.RandomZoom(0.20),
    tf.keras.layers.RandomTranslation(0.20, 0.20),
    tf.keras.layers.RandomBrightness(factor=0.30),
], name='augmentation')


def _collect_samples(split_dir, class_to_idx):
    """Walk split_dir and return (abs_paths, int_labels, rel_filenames)."""
    paths, labels, filenames = [], [], []
    for cn, ci in sorted(class_to_idx.items()):
        d = os.path.join(split_dir, cn)
        for fname in sorted(os.listdir(d)):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff')):
                paths.append(os.path.join(d, fname))
                labels.append(ci)
                filenames.append(f"{cn}/{fname}")
    return paths, labels, filenames


def convnext_preprocess(img):
    """ConvNeXt preprocessing: normalize to ImageNet stats."""
    # ConvNeXt uses standard ImageNet normalization
    img = img / 255.0
    mean = tf.constant([0.485, 0.456, 0.406], dtype=tf.float32)
    std = tf.constant([0.229, 0.224, 0.225], dtype=tf.float32)
    img = (img - mean) / std
    return img


def create_tf_datasets(data_dir, input_shape, batch_size, seed=None):
    """Build GPU-optimised tf.data pipelines for train / val / test."""
    class_names = sorted([
        d for d in os.listdir(os.path.join(data_dir, 'train'))
        if os.path.isdir(os.path.join(data_dir, 'train', d))
    ])
    class_to_idx = {cn: i for i, cn in enumerate(class_names)}
    num_classes = len(class_names)
    h, w = input_shape[:2]

    def load_and_preprocess(path, label):
        raw = tf.io.read_file(path)
        img = tf.image.decode_jpeg(raw, channels=3)
        img = tf.image.resize(img, [h, w])
        img = tf.cast(img, tf.float32)
        img = convnext_preprocess(img)
        label = tf.one_hot(label, depth=num_classes)
        return img, label

    def augment(img, lbl):
        return _augmentation(img, training=True), lbl

    def _make_split(split, training=False):
        sdir = os.path.join(data_dir, split)
        paths, labels, fns = _collect_samples(sdir, class_to_idx)
        n = len(paths)
        ds = tf.data.Dataset.from_tensor_slices((paths, labels))
        if training:
            ds = ds.shuffle(n, seed=seed, reshuffle_each_iteration=True)
        ds = ds.map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
        if training:
            ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
        ds = ds.batch(batch_size, drop_remainder=training)
        ds = ds.prefetch(AUTOTUNE)
        return ds, n, fns, labels

    train_ds, n_train, _, train_lbl = _make_split('train', training=True)
    val_ds, n_val, _, _ = _make_split('val', training=False)
    test_ds, n_test, test_fnames, test_lbl = _make_split('test', training=False)

    cw = class_weight.compute_class_weight(
        'balanced', classes=np.unique(train_lbl), y=train_lbl)
    cw_dict = dict(enumerate(cw))

    meta = SimpleNamespace(
        class_names=class_names,
        num_classes=num_classes,
        test_filenames=test_fnames,
        test_classes=np.array(test_lbl),
        n_train=n_train,
        n_val=n_val,
        n_test=n_test,
        class_weight_dict=cw_dict,
        class_weights_array=cw,  # for focal loss alpha
    )
    return train_ds, val_ds, test_ds, meta


print("Helper functions defined.")
print("  create_tf_datasets — tf.data pipeline (GPU-optimised)")
print("  convnext_preprocess — ImageNet normalization")

In [ ]:
# ========== 8. MODEL BUILDER: ConvNeXt-Base + E-EMA ========== #

def build_convnext_eema_model(input_shape, num_classes, class_weights_array,
                               steps_per_epoch, focal_gamma=2.0):
    """Build ConvNeXt-Base + E-EMA attention + Focal Loss model.
    
    Architecture:
        ConvNeXt-Base (frozen backbone)
            → E-EMA (enhanced attention)
            → GlobalAveragePooling2D
            → BatchNorm → Dense(256) → Dropout(0.5)
            → Dense(num_classes, softmax)
    """
    # --- Backbone: ConvNeXt-Base ---
    base = tf.keras.applications.ConvNeXtBase(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )
    base.trainable = False  # Freeze backbone
    
    # Get output channels from backbone
    backbone_output = base.output  # (B, H', W', C)
    backbone_channels = backbone_output.shape[-1]  # 1024 for ConvNeXt-Base
    
    print(f"  Backbone output shape: {backbone_output.shape}")
    print(f"  Backbone channels: {backbone_channels}")
    
    # --- E-EMA Attention Module ---
    # Determine groups (must divide channels evenly)
    groups = 8 if backbone_channels % 8 == 0 else 4
    x = EEMA(channels=backbone_channels, groups=groups, strip_kernel=7,
             name='e_ema_attention')(backbone_output)
    
    # --- Classification Head ---
    x = GlobalAveragePooling2D(name='global_pool')(x)
    x = BatchNormalization(name='head_bn')(x)
    x = Dense(256, activation='relu', kernel_regularizer=l2(1e-5),
              name='head_dense')(x)
    x = Dropout(0.5, name='head_dropout')(x)
    output = Dense(num_classes, activation='softmax', dtype='float32',
                   name='classifier')(x)
    
    model = Model(inputs=base.input, outputs=output,
                  name='ConvNeXtBase_EEMA')
    
    # --- Focal Loss with class-balanced alpha ---
    # Normalize class weights to sum to num_classes
    alpha = class_weights_array / class_weights_array.sum() * num_classes
    alpha = alpha.tolist()
    
    focal_loss = FocalLoss(
        gamma=focal_gamma,
        alpha=alpha,
        label_smoothing=0.1
    )
    
    # --- Optimizer: AdamW + CosineDecay ---
    lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=1e-4,
        decay_steps=steps_per_epoch * EPOCHS,
        alpha=1e-6  # minimum lr
    )
    optimizer = tf.keras.optimizers.AdamW(
        learning_rate=lr_schedule,
        weight_decay=1e-4
    )
    
    model.compile(
        optimizer=optimizer,
        loss=focal_loss,
        metrics=['accuracy'],
    )
    
    return model


print("Model builder defined: build_convnext_eema_model()")
print("  Backbone: ConvNeXt-Base (frozen)")
print("  Attention: E-EMA")
print("  Loss: Focal Loss (class-balanced)")
print("  Optimizer: AdamW + CosineDecay")

In [ ]:
# ========== 9. MULTI-RUN TRAINING EXPERIMENT ========== #
for run_idx, seed in enumerate(RANDOM_SEEDS):
    print("\n" + "="*70)
    print(f" RUN {run_idx+1}/{len(RANDOM_SEEDS)}  —  seed={seed}")
    print("="*70)

    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)

    RESULT_DIR = os.path.join(BASE_RESULT_DIR, f"run_{run_idx+1}_seed_{seed}")
    os.makedirs(RESULT_DIR, exist_ok=True)

    # --- tf.data pipeline ---
    train_ds, val_ds, test_ds, meta = create_tf_datasets(
        DATA_DIR, INPUT_SHAPE, BATCH_SIZE, seed=seed)

    steps_per_epoch = meta.n_train // BATCH_SIZE

    # --- Model ---
    model = build_convnext_eema_model(
        INPUT_SHAPE, meta.num_classes, meta.class_weights_array,
        steps_per_epoch, focal_gamma=FOCAL_GAMMA
    )
    
    if run_idx == 0:
        model.summary()

    callbacks = [
        EarlyStopping(monitor='val_loss', patience=10,
                      restore_best_weights=True, verbose=1),
        CSVLogger(os.path.join(RESULT_DIR, 'training_log.csv'), append=False),
        ModelCheckpoint(os.path.join(RESULT_DIR, 'convnext_eema_best.keras'),
                        save_best_only=True, monitor='val_loss', verbose=1),
    ]

    # --- Train ---
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        class_weight=meta.class_weight_dict,
        callbacks=callbacks,
    )

    # --- Learning curves ---
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, keys, title in zip(axes,
                                [('accuracy', 'val_accuracy'), ('loss', 'val_loss')],
                                ['Accuracy', 'Loss']):
        ax.plot(history.history[keys[0]], label='Train')
        ax.plot(history.history[keys[1]], label='Validation')
        ax.set_title(f'{title} — Run {run_idx+1} (seed={seed})')
        ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR, 'learning_curve.png'), dpi=300)
    plt.close()

    # --- Evaluation ---
    best_model = load_model(
        os.path.join(RESULT_DIR, 'convnext_eema_best.keras'),
        custom_objects={'EEMA': EEMA, 'FocalLoss': FocalLoss}
    )
    pred_probs = best_model.predict(test_ds, verbose=1)
    y_pred_run = np.argmax(pred_probs, axis=1)
    y_true_run = meta.test_classes
    class_names = meta.class_names

    report = classification_report(y_true_run, y_pred_run,
                                   target_names=class_names, output_dict=True, digits=4)
    with open(os.path.join(RESULT_DIR, 'classification_report.txt'), 'w') as f:
        f.write(classification_report(y_true_run, y_pred_run,
                                      target_names=class_names, digits=4))

    # --- Confusion matrix ---
    cm = confusion_matrix(y_true_run, y_pred_run)
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d',
                xticklabels=class_names, yticklabels=class_names, cmap='Blues', ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(f'Confusion Matrix — Run {run_idx+1} (seed={seed})')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR, 'confusion_matrix.png'), dpi=300)
    plt.close()

    # --- Store run results ---
    test_acc = np.mean(y_pred_run == y_true_run)
    all_runs_results.append({
        'run':            run_idx + 1,
        'seed':           seed,
        'accuracy':       test_acc,
        'precision':      report['weighted avg']['precision'],
        'recall':         report['weighted avg']['recall'],
        'f1_score':       report['weighted avg']['f1-score'],
        'per_class_metrics': {
            c: {'precision': report[c]['precision'],
                'recall':    report[c]['recall'],
                'f1-score':  report[c]['f1-score']}
            for c in class_names
        },
        'result_dir':     RESULT_DIR,
        'history':        history.history,
        'y_true':         y_true_run,
        'y_pred':         y_pred_run,
        'pred_probs':     pred_probs,
        'class_names':    class_names,
        'test_filenames': meta.test_filenames,
        'n_train':        meta.n_train,
        'n_val':          meta.n_val,
        'n_test':         meta.n_test,
    })

    print(f"\n  Acc={test_acc:.4f}  P={report['weighted avg']['precision']:.4f}"
          f"  R={report['weighted avg']['recall']:.4f}"
          f"  F1={report['weighted avg']['f1-score']:.4f}")

    tf.keras.backend.clear_session()

print("\n" + "="*70)
print(" ALL TRAINING RUNS COMPLETED")
print("="*70)

---
## Section 2 — Results Aggregation & Scientific Reports

Aggregate metrics across all runs, generate LaTeX tables, CSV summaries, and visualizations for publication.

In [ ]:
# ========== AGGREGATE RESULTS FROM ALL RUNS ========== #
print("\n" + "="*80)
print("📊 AGGREGATING RESULTS FROM ALL RUNS")
print("="*80 + "\n")

accuracies = [r['accuracy'] for r in all_runs_results]
precisions = [r['precision'] for r in all_runs_results]
recalls = [r['recall'] for r in all_runs_results]
f1_scores = [r['f1_score'] for r in all_runs_results]

overall_stats = {
    'Accuracy': {'mean': np.mean(accuracies), 'std': np.std(accuracies), 'values': accuracies},
    'Precision': {'mean': np.mean(precisions), 'std': np.std(precisions), 'values': precisions},
    'Recall': {'mean': np.mean(recalls), 'std': np.std(recalls), 'values': recalls},
    'F1-Score': {'mean': np.mean(f1_scores), 'std': np.std(f1_scores), 'values': f1_scores}
}

print("OVERALL METRICS ACROSS ALL RUNS:")
print("-" * 80)
for metric_name, stats in overall_stats.items():
    print(f"{metric_name:12s}: {stats['mean']:.4f} ± {stats['std']:.4f}")
    print(f"              Individual runs: {[f'{v:.4f}' for v in stats['values']]}")
print("-" * 80)

class_names = list(all_runs_results[0]['per_class_metrics'].keys())

per_class_stats = {}
for class_name in class_names:
    per_class_stats[class_name] = {}
    for metric in ['precision', 'recall', 'f1-score']:
        values = [r['per_class_metrics'][class_name][metric] for r in all_runs_results]
        per_class_stats[class_name][metric] = {
            'mean': np.mean(values), 'std': np.std(values), 'values': values
        }

print("\n✅ Statistics calculated successfully!")

In [ ]:
# ========== CREATE SCIENTIFIC REPORT TABLE ========== #
print("\n" + "="*80)
print("📋 CREATING SCIENTIFIC REPORT TABLES")
print("="*80 + "\n")

overall_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Mean': [overall_stats[m]['mean'] for m in ['Accuracy', 'Precision', 'Recall', 'F1-Score']],
    'Std': [overall_stats[m]['std'] for m in ['Accuracy', 'Precision', 'Recall', 'F1-Score']],
    'Run 1': [accuracies[0], precisions[0], recalls[0], f1_scores[0]],
    'Run 2': [accuracies[1], precisions[1], recalls[1], f1_scores[1]],
    'Run 3': [accuracies[2], precisions[2], recalls[2], f1_scores[2]]
})
overall_df['Mean ± Std'] = overall_df.apply(
    lambda row: f"{row['Mean']:.4f} ± {row['Std']:.4f}", axis=1)

print("\n📊 OVERALL PERFORMANCE METRICS (3 RUNS)")
print("="*80)
print(overall_df[['Metric', 'Mean ± Std', 'Run 1', 'Run 2', 'Run 3']].to_string(index=False))
print("="*80)

overall_df.to_csv(os.path.join(BASE_RESULT_DIR, "overall_metrics_summary.csv"), index=False)

# Per-class table
per_class_rows = []
for class_name in class_names:
    for metric in ['precision', 'recall', 'f1-score']:
        stats = per_class_stats[class_name][metric]
        per_class_rows.append({
            'Class': class_name, 'Metric': metric.capitalize(),
            'Mean': stats['mean'], 'Std': stats['std'],
            'Mean ± Std': f"{stats['mean']:.4f} ± {stats['std']:.4f}",
            'Run 1': stats['values'][0], 'Run 2': stats['values'][1], 'Run 3': stats['values'][2]
        })

per_class_df = pd.DataFrame(per_class_rows)
print("\n\n📊 PER-CLASS PERFORMANCE METRICS (3 RUNS)")
print("="*80)
for class_name in class_names:
    class_data = per_class_df[per_class_df['Class'] == class_name]
    print(f"\n{class_name}:")
    print(class_data[['Metric', 'Mean ± Std']].to_string(index=False))
print("="*80)

per_class_df.to_csv(os.path.join(BASE_RESULT_DIR, "per_class_metrics_summary.csv"), index=False)
print("\n✅ Summary tables saved to CSV files!")

In [ ]:
# ========== VISUALIZATION OF RESULTS ACROSS RUNS ========== #
print("\n" + "="*80)
print("📈 CREATING VISUALIZATIONS")
print("="*80 + "\n")

# 1. Bar plot with error bars
fig, ax = plt.subplots(figsize=(10, 6))
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
means = [overall_stats[m]['mean'] for m in metrics]
stds = [overall_stats[m]['std'] for m in metrics]
x_pos = np.arange(len(metrics))
bars = ax.bar(x_pos, means, yerr=stds, capsize=10, alpha=0.8, color='steelblue', edgecolor='black')
ax.set_xlabel('Metrics', fontsize=12, fontweight='bold')
ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title('ConvNeXt-Base + E-EMA + FocalLoss (Mean ± Std over 3 runs)', fontsize=14, fontweight='bold')
ax.set_xticks(x_pos); ax.set_xticklabels(metrics)
ax.set_ylim([0, 1.05]); ax.grid(axis='y', alpha=0.3)
for i, (mean, std) in enumerate(zip(means, stds)):
    ax.text(i, mean + std + 0.02, f'{mean:.4f}\n±{std:.4f}',
            ha='center', va='bottom', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(BASE_RESULT_DIR, "overall_metrics_barplot.png"), dpi=300, bbox_inches='tight')
plt.show()

# 2. Per-class F1-Score comparison
fig, ax = plt.subplots(figsize=(12, 6))
class_f1_means = [per_class_stats[c]['f1-score']['mean'] for c in class_names]
class_f1_stds = [per_class_stats[c]['f1-score']['std'] for c in class_names]
x_pos = np.arange(len(class_names))
bars = ax.bar(x_pos, class_f1_means, yerr=class_f1_stds, capsize=5,
              alpha=0.8, color='coral', edgecolor='black')
ax.set_xlabel('Class', fontsize=12, fontweight='bold')
ax.set_ylabel('F1-Score', fontsize=12, fontweight='bold')
ax.set_title('Per-Class F1-Score (Mean ± Std over 3 runs)', fontsize=14, fontweight='bold')
ax.set_xticks(x_pos); ax.set_xticklabels(class_names, rotation=45, ha='right')
ax.set_ylim([0, 1.05]); ax.grid(axis='y', alpha=0.3)
for i, (mean, std) in enumerate(zip(class_f1_means, class_f1_stds)):
    ax.text(i, mean + std + 0.02, f'{mean:.3f}', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(BASE_RESULT_DIR, "per_class_f1score.png"), dpi=300, bbox_inches='tight')
plt.show()

print("✅ All visualizations created and saved!")

In [ ]:
# ========== ADDITIONAL METRICS (Kappa, MCC, Balanced Acc) ========== #
print("="*70)
print("ADDITIONAL METRICS — ALL RUNS")
print("="*70)

extra_rows = []
for r in all_runs_results:
    kappa = cohen_kappa_score(r['y_true'], r['y_pred'])
    mcc = matthews_corrcoef(r['y_true'], r['y_pred'])
    bal_acc = balanced_accuracy_score(r['y_true'], r['y_pred'])
    extra_rows.append({
        'Run': r['run'], 'Seed': r['seed'],
        'Accuracy': r['accuracy'], 'Balanced Accuracy': bal_acc,
        "Cohen's Kappa": kappa, 'MCC': mcc, 'F1-Score (w)': r['f1_score'],
    })
    print(f"\nRun {r['run']} (seed={r['seed']}):")
    print(f"  Accuracy          : {r['accuracy']:.4f}")
    print(f"  Balanced Accuracy : {bal_acc:.4f}")
    print(f"  Cohen's Kappa     : {kappa:.4f}")
    print(f"  MCC               : {mcc:.4f}")
    print(f"  F1-Score (w-avg)  : {r['f1_score']:.4f}")

extra_df = pd.DataFrame(extra_rows)
num_cols = ['Accuracy', 'Balanced Accuracy', "Cohen's Kappa", 'MCC', 'F1-Score (w)']
mean_row = {'Run': 'Mean', 'Seed': '—', **{c: extra_df[c].mean() for c in num_cols}}
std_row = {'Run': 'Std', 'Seed': '—', **{c: extra_df[c].std() for c in num_cols}}
summary_extra = pd.concat([extra_df, pd.DataFrame([mean_row, std_row])], ignore_index=True)

print("\n" + "="*70)
print("SUMMARY TABLE")
print(summary_extra.to_string(index=False))

extra_df.to_csv(os.path.join(BASE_RESULT_DIR, 'additional_metrics.csv'), index=False)
summary_extra.to_csv(os.path.join(BASE_RESULT_DIR, 'additional_metrics_summary.csv'), index=False)
print("\nSaved → additional_metrics.csv, additional_metrics_summary.csv")

In [ ]:
# ========== ROC CURVES + AUC (Aggregate) ========== #
all_y_true = np.concatenate([r['y_true'] for r in all_runs_results])
all_probs = np.concatenate([r['pred_probs'] for r in all_runs_results])
cls = all_runs_results[0]['class_names']
n_cls = len(cls)
y_bin = label_binarize(all_y_true, classes=range(n_cls))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
tab_colors = plt.cm.tab10.colors
auc_scores = {}

fpr_d, tpr_d = {}, {}
for i, cname in enumerate(cls):
    fpr_d[i], tpr_d[i], _ = roc_curve(y_bin[:, i], all_probs[:, i])
    roc_auc = auc(fpr_d[i], tpr_d[i])
    auc_scores[cname] = roc_auc
    axes[0].plot(fpr_d[i], tpr_d[i], color=tab_colors[i], lw=2,
                 label=f'{cname} (AUC={roc_auc:.4f})')

axes[0].plot([0, 1], [0, 1], 'k--', lw=1)
axes[0].set(xlim=[0, 1], ylim=[0, 1.01], xlabel='FPR', ylabel='TPR',
            title='Per-Class ROC Curves (OvR) — Aggregate')
axes[0].legend(loc='lower right', fontsize=8); axes[0].grid(alpha=0.3)

# Macro-average ROC
all_fpr = np.unique(np.concatenate([fpr_d[i] for i in range(n_cls)]))
mean_tpr = np.zeros_like(all_fpr)
for i in range(n_cls):
    mean_tpr += np.interp(all_fpr, fpr_d[i], tpr_d[i])
mean_tpr /= n_cls
macro_auc = auc(all_fpr, mean_tpr)

axes[1].plot(all_fpr, mean_tpr, 'b-', lw=2, label=f'Macro-avg (AUC={macro_auc:.4f})')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
axes[1].set(xlim=[0, 1], ylim=[0, 1.01], xlabel='FPR', ylabel='TPR',
            title='Macro-Average ROC Curve')
axes[1].legend(fontsize=10); axes[1].grid(alpha=0.3)

plt.suptitle('ROC Curves — ConvNeXt-Base + E-EMA + FocalLoss (Aggregate)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(BASE_RESULT_DIR, 'roc_curves.png'), dpi=300, bbox_inches='tight')
plt.show()

print(f"\nMacro-average AUC: {macro_auc:.4f}")
auc_df = pd.DataFrame({'Class': list(auc_scores.keys()), 'AUC': list(auc_scores.values())})
auc_df.to_csv(os.path.join(BASE_RESULT_DIR, 'auc_scores.csv'), index=False)
print("Saved → roc_curves.png, auc_scores.csv")

In [ ]:
# ========== TRAINING CONVERGENCE ANALYSIS ========== #
colors3 = ['steelblue', 'coral', 'seagreen']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for r, c in zip(all_runs_results, colors3):
    axes[0, 0].plot(r['history']['val_accuracy'], color=c,
                    label=f"Run {r['run']} (seed={r['seed']})", lw=2)
axes[0, 0].set_title('Validation Accuracy — All Runs', fontweight='bold')
axes[0, 0].set_xlabel('Epoch'); axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].legend(); axes[0, 0].grid(alpha=0.3)

for r, c in zip(all_runs_results, colors3):
    axes[0, 1].plot(r['history']['val_loss'], color=c,
                    label=f"Run {r['run']} (seed={r['seed']})", lw=2)
axes[0, 1].set_title('Validation Loss — All Runs', fontweight='bold')
axes[0, 1].set_xlabel('Epoch'); axes[0, 1].set_ylabel('Loss')
axes[0, 1].legend(); axes[0, 1].grid(alpha=0.3)

best_epochs = [np.argmin(r['history']['val_loss']) + 1 for r in all_runs_results]
run_labels = [f"Run {r['run']}\n(seed={r['seed']})" for r in all_runs_results]
bars = axes[1, 0].bar(run_labels, best_epochs, color=colors3[:len(all_runs_results)],
                      edgecolor='black', alpha=0.85)
axes[1, 0].set_title('Best Epoch per Run', fontweight='bold')
axes[1, 0].set_ylabel('Epoch'); axes[1, 0].grid(axis='y', alpha=0.3)
for bar, ep in zip(bars, best_epochs):
    axes[1, 0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                    str(ep), ha='center', fontweight='bold')

for r, c in zip(all_runs_results, colors3):
    gap = np.array(r['history']['accuracy']) - np.array(r['history']['val_accuracy'])
    axes[1, 1].plot(gap, color=c, label=f"Run {r['run']}", lw=2)
axes[1, 1].axhline(0, color='black', linestyle='--', lw=1)
axes[1, 1].set_title('Train−Val Accuracy Gap (Overfitting)', fontweight='bold')
axes[1, 1].set_xlabel('Epoch'); axes[1, 1].set_ylabel('Gap')
axes[1, 1].legend(); axes[1, 1].grid(alpha=0.3)

plt.suptitle('Training Convergence — ConvNeXt-Base + E-EMA + FocalLoss',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(BASE_RESULT_DIR, 'convergence_analysis.png'), dpi=300, bbox_inches='tight')
plt.show()
print("Saved → convergence_analysis.png")

---
## Section 3 — Single-Run Deep-Dive Analysis

Grad-CAM, t-SNE, error analysis for the selected run.

In [ ]:
# ========== SELECT RUN FOR SINGLE-RUN ANALYSIS ========== #
SELECTED_RUN = len(RANDOM_SEEDS)   # default: last run

run_data = all_runs_results[SELECTED_RUN - 1]
RESULT_DIR = run_data['result_dir']
y_true = run_data['y_true']
y_pred = run_data['y_pred']
pred_probs = run_data['pred_probs']
class_names = run_data['class_names']
test_acc = run_data['accuracy']
n_train = run_data['n_train']
n_val = run_data['n_val']
n_test = run_data['n_test']
test_filenames = run_data['test_filenames']
test_dir = os.path.join(DATA_DIR, 'test')
history = SimpleNamespace(history=run_data['history'])

# Reload best model
model = load_model(
    os.path.join(RESULT_DIR, 'convnext_eema_best.keras'),
    custom_objects={'EEMA': EEMA, 'FocalLoss': FocalLoss}
)

# Rebuild test pipeline
_, _, test_ds, _ = create_tf_datasets(
    DATA_DIR, INPUT_SHAPE, BATCH_SIZE, seed=run_data['seed'])

print(f"Analysing Run {SELECTED_RUN} (seed={run_data['seed']})")
print(f"  Classes  : {class_names}")
print(f"  Train/Val/Test : {n_train}/{n_val}/{n_test}")
print(f"  Accuracy : {run_data['accuracy']:.4f}")
print(f"  F1-Score : {run_data['f1_score']:.4f}")

In [ ]:
# ========== GRAD-CAM VISUALIZATION ========== #

def make_gradcam_heatmap(img_array, grad_model, pred_index=None):
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]
    grads = tape.gradient(class_channel, conv_out)
    pooled = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = conv_out[0] @ pooled[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

def overlay_heatmap(orig_img_array, heatmap, alpha=0.4):
    heatmap_uint8 = np.uint8(255 * heatmap)
    jet_colors = plt.cm.jet(np.arange(256))[:, :3]
    jet_heatmap = jet_colors[heatmap_uint8]
    jet_heatmap = tf.keras.preprocessing.image.array_to_img(jet_heatmap)
    jet_heatmap = jet_heatmap.resize((orig_img_array.shape[1], orig_img_array.shape[0]))
    jet_heatmap = img_to_array(jet_heatmap)
    superimposed = jet_heatmap * alpha + orig_img_array
    return np.clip(superimposed / superimposed.max(), 0, 1)

# Find last Conv2D layer for Grad-CAM
last_conv_name = [l.name for l in model.layers
                  if isinstance(l, tf.keras.layers.Conv2D)][-1]
print(f"Last conv layer: {last_conv_name}")

grad_model = Model(inputs=model.inputs,
                   outputs=[model.get_layer(last_conv_name).output, model.output])

n_cls = len(class_names)
fig, axes = plt.subplots(n_cls, 3, figsize=(12, 4 * n_cls))
if n_cls == 1:
    axes = axes[np.newaxis, :]

for ci, cname in enumerate(class_names):
    ok_idx = np.where((y_true == ci) & (y_pred == ci))[0]
    idx = ok_idx[0] if len(ok_idx) > 0 else np.where(y_true == ci)[0][0]
    fpath = os.path.join(test_dir, test_filenames[idx])

    img_orig = load_img(fpath, target_size=INPUT_SHAPE[:2])
    img_arr = img_to_array(img_orig)
    # Preprocess for ConvNeXt
    img_proc = img_arr.copy() / 255.0
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img_proc = (img_proc - mean) / std
    img_proc = np.expand_dims(img_proc, 0).astype(np.float32)

    heatmap = make_gradcam_heatmap(img_proc, grad_model, pred_index=ci)
    overlay = overlay_heatmap(img_arr, heatmap)

    axes[ci, 0].imshow(img_orig); axes[ci, 0].axis('off')
    axes[ci, 0].set_title(f'Original\nClass: {cname}', fontsize=9)
    axes[ci, 1].imshow(heatmap, cmap='jet'); axes[ci, 1].axis('off')
    axes[ci, 1].set_title('Grad-CAM Heatmap', fontsize=9)
    axes[ci, 2].imshow(overlay); axes[ci, 2].axis('off')
    conf = pred_probs[idx][y_pred[idx]]
    axes[ci, 2].set_title(f'Overlay\nConf={conf:.2%}', fontsize=9)

plt.suptitle(f'Grad-CAM — ConvNeXt-Base + E-EMA (Run {SELECTED_RUN})',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, 'gradcam_visualization.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Saved → gradcam_visualization.png")

In [ ]:
# ========== t-SNE FEATURE EMBEDDING ========== #
gap_layer_name = [l.name for l in model.layers if 'global' in l.name and 'pool' in l.name][0]
feature_extractor = Model(inputs=model.input,
                          outputs=model.get_layer(gap_layer_name).output)

print("Extracting features from test set...")
features = feature_extractor.predict(test_ds, verbose=1)

print("Computing t-SNE embedding...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
features_2d = tsne.fit_transform(features)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
palette = plt.cm.tab10.colors

for ci, cname in enumerate(class_names):
    mask = y_true == ci
    axes[0].scatter(features_2d[mask, 0], features_2d[mask, 1],
                    c=[palette[ci]], label=cname, alpha=0.7, s=20)
axes[0].set_title(f't-SNE — True Labels (Run {SELECTED_RUN})', fontweight='bold')
axes[0].legend(markerscale=2, fontsize=9); axes[0].grid(alpha=0.3)

cm_mask = y_true == y_pred
axes[1].scatter(features_2d[cm_mask, 0], features_2d[cm_mask, 1],
                c='steelblue', label=f'Correct ({cm_mask.sum()})', alpha=0.6, s=20)
axes[1].scatter(features_2d[~cm_mask, 0], features_2d[~cm_mask, 1],
                c='red', label=f'Misclassified ({(~cm_mask).sum()})',
                alpha=0.9, s=50, marker='x')
axes[1].set_title('t-SNE — Correct vs Misclassified', fontweight='bold')
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

plt.suptitle(f't-SNE — ConvNeXt-Base + E-EMA (Run {SELECTED_RUN})',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, 'tsne_embedding.png'), dpi=300, bbox_inches='tight')
plt.show()
print("Saved → tsne_embedding.png")

In [ ]:
# ========== ERROR ANALYSIS ========== #
print("="*70)
print("ERROR ANALYSIS")
print("="*70)

confusion_counts = defaultdict(int)
for yt, yp in zip(y_true, y_pred):
    if yt != yp:
        confusion_counts[(class_names[yt], class_names[yp])] += 1

print(f"\nTotal misclassifications: {sum(confusion_counts.values())} / {len(y_true)}")
print(f"Overall accuracy: {np.mean(y_true == y_pred):.4f}\n")
print("Most common confused pairs (True → Predicted):")
for (tc, pc), cnt in sorted(confusion_counts.items(), key=lambda x: -x[1])[:10]:
    pct = cnt / np.sum(y_true == class_names.index(tc)) * 100
    print(f"  {tc:<22} → {pc:<22}  {cnt:3d} samples  ({pct:.1f}% of class)")

# Confidence distribution
correct_mask = y_true == y_pred
correct_conf = pred_probs[correct_mask][np.arange(correct_mask.sum()), y_pred[correct_mask]]
wrong_conf = pred_probs[~correct_mask][np.arange((~correct_mask).sum()), y_pred[~correct_mask]]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(correct_conf, bins=20, color='steelblue', alpha=0.7, edgecolor='black',
             label=f'Correct (n={len(correct_conf)})')
axes[0].hist(wrong_conf, bins=20, color='salmon', alpha=0.7, edgecolor='black',
             label=f'Wrong (n={len(wrong_conf)})')
axes[0].axvline(0.5, color='black', linestyle='--', lw=1)
axes[0].set_title('Prediction Confidence Distribution', fontweight='bold')
axes[0].set_xlabel('Confidence'); axes[0].set_ylabel('Count')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Per-class accuracy
class_acc = [(cn, np.mean(y_pred[y_true == ci] == ci), (y_true == ci).sum())
             for ci, cn in enumerate(class_names)]
class_acc.sort(key=lambda x: x[1])
names, accs, counts = zip(*class_acc)
colors_bar = plt.cm.RdYlGn(np.array(accs))
bars = axes[1].barh(names, accs, color=colors_bar, edgecolor='black', height=0.6)
axes[1].set_title('Per-Class Accuracy (sorted)', fontweight='bold')
axes[1].set_xlabel('Accuracy'); axes[1].set_xlim([0, 1.15])
axes[1].grid(axis='x', alpha=0.3)
for bar, acc, n in zip(bars, accs, counts):
    axes[1].text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                 f'{acc:.3f} (n={n})', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, 'error_analysis.png'), dpi=300, bbox_inches='tight')
plt.show()
print("Saved → error_analysis.png")

In [ ]:
# ========== ZIP ALL RESULTS ========== #
import shutil

print("\n" + "="*80)
print("🗜️  CREATING COMPLETE ARCHIVE")
print("="*80 + "\n")

zip_output_path = "/kaggle/working/ConvNeXtBase_EEMA_FocalLoss_Complete"
print(f"Source: {BASE_RESULT_DIR}")
print(f"Output: {zip_output_path}.zip")

shutil.make_archive(zip_output_path, 'zip', BASE_RESULT_DIR)

zip_size = os.path.getsize(f"{zip_output_path}.zip") / (1024*1024)
print(f"\n✅ Complete report archived successfully!")
print(f"📦 Archive size: {zip_size:.2f} MB")
print(f"📍 Location: {zip_output_path}.zip")
print("\n" + "="*80)
print("🎉 ALL DONE!")
print("="*80)